In [27]:
from pathlib import Path

import torch
from torch import nn

from omegaconf import DictConfig, OmegaConf
from omegaconf import OmegaConf as om

import sys
sys.path.append("../..")

from ModernBERT.src import flex_bert as flex_bert_module
from ModernBERT.src import hf_bert as hf_bert_module
from ModernBERT.src import mosaic_bert as mosaic_bert_module
from ModernBERT.src.bert_layers.model import init_mlm_model_from_pretrained
from ModernBERT.src.bert_layers.configuration_bert import FlexBertConfig

from composer.utils.checkpoint import _ensure_valid_checkpoint

def build_model(cfg: DictConfig):
	if cfg.name == "hf_bert":
		return hf_bert_module.create_hf_bert_mlm(
			pretrained_model_name=cfg.pretrained_model_name,
			use_pretrained=cfg.get("use_pretrained", None),
			model_config=cfg.get("model_config", None),
			tokenizer_name=cfg.get("tokenizer_name", None),
			gradient_checkpointing=cfg.get("gradient_checkpointing", None),
		)
	elif cfg.name == "mosaic_bert":
		return mosaic_bert_module.create_mosaic_bert_mlm(
			pretrained_model_name=cfg.pretrained_model_name,
			pretrained_checkpoint=cfg.get("pretrained_checkpoint", None),
			model_config=cfg.get("model_config", None),
			tokenizer_name=cfg.get("tokenizer_name", None),
			gradient_checkpointing=cfg.get("gradient_checkpointing", None),
		)
	elif cfg.name == "flex_bert":
		return flex_bert_module.create_flex_bert_mlm(
			pretrained_model_name=cfg.pretrained_model_name,
			pretrained_checkpoint=cfg.get("pretrained_checkpoint", None),
			model_config=cfg.get("model_config", None),
			tokenizer_name=cfg.get("tokenizer_name", None),
			gradient_checkpointing=cfg.get("gradient_checkpointing", None),
			recompute_metric_loss=cfg.get("recompute_metric_loss", False),
			disable_train_metrics=cfg.get("disable_train_metrics", False),
		)
	else:
		raise ValueError(f"Not sure how to build model with name={cfg.name}")

def init_from_checkpoint(cfg: DictConfig, new_model: nn.Module):
	print(f"Initializing model from checkpoint {cfg.checkpoint_run_name}")
	# checkpoint_cfg = Path(cfg.checkpoint_cfg)
	# assert checkpoint_cfg.exists(), f"Checkpoint config {checkpoint_cfg} does not exist"
	# pretrained_cfg = om.load(checkpoint_cfg)
	pretrained_cfg = cfg

	pretrained_model = build_model(cfg.model)
	n_params = sum(p.numel() for p in pretrained_model.parameters())

	checkpoint_filepath = Path(cfg.checkpoint_load_path) / f"{cfg.checkpoint_run_name}" / "latest-rank0.pt"
	assert checkpoint_filepath.exists(), f"Checkpoint {checkpoint_filepath} does not exist"
	state = torch.load(_ensure_valid_checkpoint(checkpoint_filepath), map_location="cpu")

	state_dict = state.get("state", {})
	model_state = state_dict.get("model", {})
	assert len(model_state) > 0, "Model state is empty, please check the checkpoint and checkpoint path"

	pretrained_model.load_state_dict(model_state)

	if isinstance(pretrained_cfg.model.model_config, DictConfig):
		model_config = OmegaConf.to_container(pretrained_cfg.model.model_config, resolve=True)
	pretrained_config = FlexBertConfig.from_pretrained(pretrained_cfg.model.pretrained_model_name, **model_config)

	init_mlm_model_from_pretrained(
		config=pretrained_config,
		pretrained_model=pretrained_model.model,
		new_model=new_model.model,
		mode=cfg.get("mode", "tile_weights_from_middle"),
	)
	print(f"Initalized model from checkpoint {cfg.checkpoint_run_name} with {n_params=:.4e} parameters")


In [28]:
# cfg_path = "/disk/10tb/home/fishman/DNALM/ModernBERT/runs/cfg.yaml"
cfg_path = "./data/ModernGena_t2t_test_pretrain_model.yaml"
yaml_cfg = om.load(cfg_path)
model = build_model(yaml_cfg.model)
n_params = sum(p.numel() for p in model.parameters())
print (n_params)
checkpoint_filepath = "/disk/10tb/home/fishman/DNALM/ModernBERT/runs/moderngena_t2t_testrun/latest-rank0.pt"
checkpoint_filepath = Path(checkpoint_filepath)
assert checkpoint_filepath.exists(), f"Checkpoint {checkpoint_filepath} does not exist"
state = torch.load(_ensure_valid_checkpoint(checkpoint_filepath), map_location="cpu")

state_dict = state.get("state", {})
model_state = state_dict.get("model", {})
assert len(model_state) > 0, "Model state is empty, please check the checkpoint and checkpoint path"

model.load_state_dict(model_state)


SDPA attention is being used without an attention mask. Including padding in the  attention calculation may cause differences from the Flash Attention implementation.


ValueError: Sliding window is not implemented for the PyTorch SDPA path. Use the FA2 backend.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("AIRI-Institute/gena-lm-bert-base-t2t")
inputs = tokenizer("ATTATATAGATGGACATGACATAGCAGTAGC")

/disk/10tb/home/fishman/miniconda3/envs/bert24/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [26]:
model.cuda()
model.eval()
with torch.no_grad():
    # Convert inputs dict to tensor on device
    model_inputs = {
        'input_ids': torch.tensor([inputs['input_ids']]).cuda(),
        'attention_mask': torch.tensor([inputs['attention_mask']]).cuda()
    }
    outputs = model(model_inputs)
    logits = outputs.logits
    preds = torch.argmax(logits, dim=-1)
    print(preds)

RuntimeError: FlashAttention only supports Ampere GPUs or newer.